# Gold read optimization: evidence for the report
The Gold tables hold only about 30,000 rows, so on the real data there is almost nothing for
`OPTIMIZE` or Liquid Clustering to improve. To produce honest before/after evidence, this notebook:

1. Builds two **scratch copies** of `fact_commits` in a separate schema (`gold_bench`), scaled up to
   several million rows and loaded as many small appends. That mimics an enterprise pipeline that
   MERGEs every hour and leaves many small files.
   - `commits_unclustered`: plain Delta table
   - `commits_clustered`: `CLUSTER BY (repository_key, commit_date_key)`, the same keys as the real fact
2. Times two typical dashboard queries before and after `OPTIMIZE`.
3. Shows the `OPTIMIZE` history of the real Gold tables and a query plan with pushed-down filters.

**Fair-test note:** both scratch tables use `delta.targetFileSize = 2mb`. A real table of this shape
would be hundreds of MB and split into many files; the small target reproduces that file count at demo
scale. State this in the report.

Runtime is a few minutes on Serverless. Nothing in the `gold` schema is touched.

In [0]:
%run ./common

In [0]:
import time
import statistics

import pandas as pd
from pyspark.sql import functions as F

spark.conf.set('spark.sql.session.timeZone', 'UTC')

GOLD = f'{CATALOG}.gold'
BENCH = f'{CATALOG}.gold_bench'
assert BENCH.endswith('gold_bench')   # safety: only ever drop the scratch schema

APPENDS = 60              # number of small writes (think: 60 hourly pipeline runs)
REPS_PER_APPEND = 15      # copies of fact_commits per write
TARGET_FILE_SIZE = '2mb'  # emulate production volume at demo scale
TABLES = ['commits_unclustered', 'commits_clustered']

spark.sql(f'DROP SCHEMA IF EXISTS {BENCH} CASCADE')
spark.sql(f'CREATE SCHEMA {BENCH}')

COLUMNS = ('record_key STRING, repository_key BIGINT, contributor_key BIGINT, '
           'commit_date_key INT, message_length INT, commit_count INT')
PROPS = f"TBLPROPERTIES ('delta.targetFileSize' = '{TARGET_FILE_SIZE}')"

spark.sql(f'CREATE TABLE {BENCH}.commits_unclustered ({COLUMNS}) USING DELTA {PROPS}')
spark.sql(f'CREATE TABLE {BENCH}.commits_clustered ({COLUMNS}) USING DELTA '
          f'CLUSTER BY (repository_key, commit_date_key) {PROPS}')
print('Scratch tables created in', BENCH)

## 1. Load the scratch tables as many small appends

In [0]:
base = (spark.table(f'{GOLD}.fact_commits')
        .select('record_key', 'repository_key', 'contributor_key',
                'commit_timestamp', 'message_length', 'commit_count'))
base_rows = base.count()
assert base_rows > 0, 'fact_commits is empty: run gold_load first'

t_start = time.perf_counter()
for i in range(APPENDS):
    reps = spark.range(i * REPS_PER_APPEND, (i + 1) * REPS_PER_APPEND).withColumnRenamed('id', 'rep')
    batch = (base.crossJoin(F.broadcast(reps)).select(
        F.concat_ws('#', 'record_key', F.col('rep').cast('string')).alias('record_key'),
        'repository_key', 'contributor_key',
        # spread the dates over about 3 more years so there is something to skip
        F.expr("CAST(date_format(date_add(to_date(commit_timestamp), CAST(rep % 900 AS INT)), "
               "'yyyyMMdd') AS INT)").alias('commit_date_key'),
        'message_length', 'commit_count'))
    for table in TABLES:
        batch.write.mode('append').insertInto(f'{BENCH}.{table}')
    if (i + 1) % 10 == 0:
        print(f'  {i + 1}/{APPENDS} appends done ({time.perf_counter() - t_start:.0f}s)', flush=True)

counts = {t: spark.table(f'{BENCH}.{t}').count() for t in TABLES}
print('Rows per table:', counts)
assert len(set(counts.values())) == 1, 'Both tables must hold identical data'
assert counts[TABLES[0]] == base_rows * APPENDS * REPS_PER_APPEND

## 2. Measurement helper
Two queries that mirror the dashboard: a selective filter and a monthly trend for one repository.
Each query runs 5 times per table, interleaved to avoid order bias; the first run is a warm-up and is
excluded from the median.

In [0]:
repo_row = (spark.table(f'{GOLD}.dim_repository').filter("repository_name = 'apache/spark'").first()
            or spark.table(f'{GOLD}.dim_repository').first())
repo_key = repo_row['repository_key']
print('Benchmark repository:', repo_row['repository_name'], repo_key)

QUERIES = {
    'Q1 one repo, one month':
        "SELECT COUNT(*) AS commits FROM {t} "
        f"WHERE repository_key = {repo_key} AND commit_date_key BETWEEN 20260101 AND 20260131",
    'Q2 one repo, monthly trend':
        "SELECT commit_date_key DIV 100 AS year_month, COUNT(*) AS commits FROM {t} "
        f"WHERE repository_key = {repo_key} GROUP BY 1 ORDER BY 1",
}


def measure(stage, runs=5):
    timings = {(t, q): [] for t in TABLES for q in QUERIES}
    results = {}
    for _ in range(runs):
        for q, template in QUERIES.items():
            for t in TABLES:
                sql = template.format(t=f'{BENCH}.{t}')
                t0 = time.perf_counter()
                out = spark.sql(sql).collect()
                timings[(t, q)].append(time.perf_counter() - t0)
                results[(t, q)] = [tuple(r) for r in out]
    for q in QUERIES:   # both tables must return the same answer
        assert results[(TABLES[0], q)] == results[(TABLES[1], q)], f'Results differ for {q}'
    rows = []
    for (t, q), values in timings.items():
        d = spark.sql(f'DESCRIBE DETAIL {BENCH}.{t}').first()
        rows.append({'stage': stage, 'table': t, 'query': q,
                     'files': d['numFiles'], 'size_mb': round(d['sizeInBytes'] / 1048576, 1),
                     'median_s': round(statistics.median(values[1:]), 2),
                     'best_s': round(min(values), 2)})
    return pd.DataFrame(rows)

## 3. Before OPTIMIZE (many small files)

In [0]:
before = measure('1 before OPTIMIZE')
display(before)

## 4. Run OPTIMIZE on both tables
The unclustered table is only **compacted** (small files merged). The clustered table is compacted
**and** its rows are physically sorted by the clustering keys, which is what enables file skipping.

In [0]:
opt = {}
for t in TABLES:
    t0 = time.perf_counter()
    row = spark.sql(f'OPTIMIZE {BENCH}.{t}').first()
    metrics = row['metrics'].asDict()
    opt[t] = {'files_removed': metrics.get('numFilesRemoved'),
              'files_added': metrics.get('numFilesAdded'),
              'seconds': round(time.perf_counter() - t0, 1)}
display(pd.DataFrame(opt).T.reset_index().rename(columns={'index': 'table'}))

## 5. After OPTIMIZE

In [0]:
after = measure('2 after OPTIMIZE')
display(after)

## 6. Summary table for the report

In [0]:
summary = before.merge(after, on=['table', 'query'], suffixes=('_before', '_after'))
summary['speedup_x'] = (summary['median_s_before'] / summary['median_s_after']).round(1)
summary = summary[['table', 'query', 'files_before', 'files_after',
                   'median_s_before', 'median_s_after', 'speedup_x']]
display(summary)

print('Screenshot for the report: open Query History, choose the Q1 run on each table and compare')
print('"Files read" vs "Files pruned" in the query profile. The clustered table should read far fewer files.')

## 7. The real Gold tables: clustering and OPTIMIZE history
`run_gold` calls `OPTIMIZE` automatically after each refresh when a table has been written since its
last OPTIMIZE. The history below is the proof that it ran.

In [0]:
CLUSTERED = ['fact_commits', 'fact_issues', 'fact_pull_requests',
             'fact_repo_daily_activity', 'fact_contributor_activity']

state = []
for t in CLUSTERED:
    d = spark.sql(f'DESCRIBE DETAIL {GOLD}.{t}').first()
    state.append({'table': t, 'clustering_columns': list(d['clusteringColumns']),
                  'files': d['numFiles'], 'size_kb': round(d['sizeInBytes'] / 1024)})
display(pd.DataFrame(state))

history = None
for t in CLUSTERED:
    h = (spark.sql(f'DESCRIBE HISTORY {GOLD}.{t}').filter("operation = 'OPTIMIZE'")
         .select(F.lit(t).alias('table'), 'version', 'timestamp',
                 F.col('operationMetrics')['numRemovedFiles'].alias('files_removed'),
                 F.col('operationMetrics')['numAddedFiles'].alias('files_added')))
    history = h if history is None else history.unionByName(h)
display(history.orderBy('table', F.col('version').desc()))

## 8. Query plan with pushed-down filters (for the Spark UI section)
Look for `PushedFilters` / `DataFilters` on the `fact_issues` scan and the broadcast joins to the
dimensions.

In [0]:
plan = spark.sql(f'''
EXPLAIN FORMATTED
SELECT r.repository_name, d.year_month, AVG(i.hours_to_close) AS avg_hours_to_close
FROM {GOLD}.fact_issues i
JOIN {GOLD}.dim_repository r ON i.repository_key = r.repository_key
JOIN {GOLD}.dim_date d       ON i.closed_date_key = d.date_key
WHERE i.is_closed AND r.repository_name = 'apache/spark'
GROUP BY r.repository_name, d.year_month
''').first()[0]
print(plan)

## 9. Clean up the scratch schema
Set `CLEAN_UP = True` and run this cell once you have your screenshots and numbers.

In [0]:
CLEAN_UP = False

if CLEAN_UP:
    spark.sql(f'DROP SCHEMA IF EXISTS {BENCH} CASCADE')
    print('Dropped', BENCH)
else:
    print('Scratch tables kept. Set CLEAN_UP = True to drop', BENCH)